In [ ]:
pip install deeppavlov #переустанавливаю занава так как возникли проблемы с файлами

In [ ]:
!pip install pymorphy2 pymorphy2-dicts-ru

In [1]:
import pandas as pd
import pymorphy2
import nltk
from nltk.stem import WordNetLemmatizer
from deeppavlov import build_model
import torch
import os

# отключение GPU из за несавместимости версии torch и видео карты
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
torch.cuda.is_available = lambda: False

print("GPU принудительно отключён. Работаем только на CPU.\n")

# настраиваем лимитизаторы
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer_en = WordNetLemmatizer()

def lemmatize_en(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    return " ".join([lemmatizer_en.lemmatize(w) for w in text.lower().split()])

morph = pymorphy2.MorphAnalyzer()

def lemmatize_ru(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    return " ".join([morph.parse(w)[0].normal_form for w in text.split()])

# загрузка моделей
print("Загрузка моделей DeepPavlov...")

model_en = build_model("insults_kaggle_bert", download=True)
model_ru = build_model("rusentiment_bert", download=True)

print("Модели загружены.")

# Безопасное перемещение на CPU
try:
    if hasattr(model_en, 'model'):
        model_en.model.to('cpu')
    elif hasattr(model_en, 'pipe'):
        for component in model_en.pipe:
            if hasattr(component, 'model'):
                component.model.to('cpu')
except:
    pass

try:
    if hasattr(model_ru, 'model'):
        model_ru.model.to('cpu')
    elif hasattr(model_ru, 'pipe'):
        for component in model_ru.pipe:
            if hasattr(component, 'model'):
                component.model.to('cpu')
except:
    pass

print("Модели успешно загружены и работают ")

GPU принудительно отключён. Работаем только на CPU.



I:\SNAKS\envs\py310\lib\site-packages\pymorphy2\analyzer.py:114: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Загрузка моделей DeepPavlov...


2026-04-09 00:02:23.774 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/deeppavlov_data/classifiers/insults_kaggle_torch_bert_v5.tar.gz download because of matching hashes
2026-04-09 00:02:44.825 INFO in 'deeppavlov.core.data.utils'['utils'] at line 97: Downloading from http://files.deeppavlov.ai/datasets/insults_data.tar.gz to C:\Users\vovur\.deeppavlov\insults_data.tar.gz
100%|██████████| 682k/682k [00:00<00:00, 1.91MB/s]
2026-04-09 00:02:46.131 INFO in 'deeppavlov.core.data.utils'['utils'] at line 284: Extracting C:\Users\vovur\.deeppavlov\insults_data.tar.gz archive into C:\Users\vovur\.deeppavlov\downloads
I:\SNAKS\envs\py310\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I:\SNAKS\envs\py310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resu

Модели загружены.
Модели успешно загружены и работают 


In [2]:
print("анализ английского датасета")

df_en = pd.read_csv("toxic_comments (1).csv", on_bad_lines='skip', engine='python', nrows=500)
df_en = df_en.dropna(subset=['text']).reset_index(drop=True)
df_en_sample = df_en.head(20).copy()

df_en_sample['lemmatized'] = df_en_sample['text'].apply(lemmatize_en)
texts_en = df_en_sample['lemmatized'].tolist()

df_en_sample['prediction'] = model_en(texts_en)

display(df_en_sample[['text', 'toxic', 'lemmatized', 'prediction']])

анализ английского датасета


,text,toxic,lemmatized,prediction
0,Explanation\nWhy the edits made under my usern...,0,explanation why the edits made under my userna...,Not Insult
1,D'aww! He matches this background colour I'm s...,0,d'aww! he match this background colour i'm see...,Not Insult
2,"Hey man, I'm really not trying to edit war. It...",0,"hey man, i'm really not trying to edit war. it...",Not Insult
3,"""\nMore\nI can't make any real suggestions on ...",0,""" more i can't make any real suggestion on imp...",Not Insult
4,"You, sir, are my hero. Any chance you remember...",0,"you, sir, are my hero. any chance you remember...",Not Insult
5,"""\n\nCongratulations from me as well, use the ...",0,""" congratulation from me a well, use the tool ...",Not Insult
6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1,cocksucker before you piss around on my work,Insult
7,Your vandalism to the Matt Shirvington article...,0,your vandalism to the matt shirvington article...,Not Insult
8,Sorry if the word 'nonsense' was offensive to ...,0,sorry if the word 'nonsense' wa offensive to y...,Not Insult
9,alignment on this subject and which are contra...,0,alignment on this subject and which are contra...,Not Insult


In [3]:
print("анализ русского датасета")

df_ru = pd.read_csv("rusentitweet_full.csv", on_bad_lines='skip', engine='python', nrows=500)
df_ru = df_ru[df_ru['label'] != 'skip'].reset_index(drop=True)
df_ru_sample = df_ru.head(20).copy()

df_ru_sample['lemmatized'] = df_ru_sample['text'].apply(lemmatize_ru)
texts_ru = df_ru_sample['lemmatized'].tolist()

df_ru_sample['prediction'] = model_ru(texts_ru)

display(df_ru_sample[['text', 'label', 'lemmatized', 'prediction']])

анализ русского датасета


,text,label,lemmatized,prediction
0,велл они всё равно что мусор так что ничего с...,negative,велла они всё равно что мусор так что ничего с...,negative
1,"""трезвая жизнь какая-то такая стрёмная""\r\n(с)...",negative,"""трезвый жизнь какой-то такой стрёмная"" (с) ар...",neutral
2,Ой какие неожиданные результаты 🤭 https://t.co...,neutral,ой какой неожиданный результат 🤭 https://t.co/...,neutral
3,@Shvonder_chief @dimsmirnov175 На заборе тоже ...,neutral,@shvonder_chief @dimsmirnov175 на забор тоже н...,neutral
4,@idkwhht мы тоже мебельная компания уджина😳😳😳,neutral,@idkwhht мы тоже мебельный компания уджина😳😳😳,positive
5,Счастья здоровья 10 классникам https://t.co/M9...,speech,счастие здоровье 10 классник https://t.co/m9vu...,neutral
6,@dntbliev НЕ ПАЛИ.,neutral,@dntbliev не пали.,negative
7,@BTS_twt ты такой красивый 😭😭😭🥺💓,positive,@bts_twt ты такой красивый 😭😭😭🥺💓,positive
8,"@Ladyzchensk Цыган , хуле ...",negative,"@ladyzchensk цыган , хула ...",negative
9,@nikogdanahui1 С днём рождения 🥳 \r\nУспехов и...,speech,@nikogdanahui1 с день рождение 🥳 успех и побед...,speech
